# IAPR: Final Project ‒ UNO Game State Recovery


**Group ID:** 46

**Author 1 (sciper):** Lemonnier Théo (312284)  
**Author 2 (sciper):** Desnos Edgar (388960)   
**Author 3 (sciper):** Anthonin Duval (xxxxx)

## Imports and Configurations

## Card Segmentation and Cropping

Before classifying any symbol, we need to isolate the individual cards from the rest of the scene. We implemented a fully classical segmentation pipeline (`src/segment_cards.py`) that works on both the white and the noisy leafy backgrounds.

The image is first converted to the HLS color space and only the saturation channel is kept. Card bodies have a strongly saturated color compared to the background, so a simple Gaussian blur followed by an `inRange` threshold on the saturation channel produces a clean binary mask. The threshold is slightly lower on the white background (`15`) than on the noisy one (`30`) because cards stand out more easily against the white. A morphological closing with a 5x5 elliptical kernel fills in small holes caused by symbols and digits printed on the card.

We then extract external contours and discard anything with an area below 40000 pixels — this removes tokens, hands and other noise. Each remaining contour is simplified with `cv2.approxPolyDP` using a 3% perimeter tolerance, which collapses the card outline to a 4-point quadrilateral.

From each quadrilateral we derive three things used downstream:
- the **centroid**, used to assign the card to a player zone or to the center,
- a **body mask** (with a small erosion to stay safely inside the card), used to vote on the card color from the HSV color masks,
- two **corner crops** obtained by warping the card to a canonical 200x310 rectangle and resizing the top-left and bottom-right corner regions to 80x80 RGB patches, which are the exact input format expected by the CNN.

## Color Filtering

Color classification is also fully classical (`src/color_classifier.py`, `src/detect_reference.py`). For each input image we compute five HSV masks — Yellow, Red, Blue, Green and Black — using hand-tuned ranges. Red wraps around the HSV hue cylinder, so it is built as the union of two separate ranges (low hue and high hue).

To label a card we restrict each color mask to the card's body mask returned by the segmenter, count the number of mask pixels that fall inside, and pick the color with the most votes. This is robust to the printed symbol pixels because they only account for a small fraction of the card body. The four returned labels are mapped to the submission alphabet `{r, g, b, y}`.

Two symbol classes (`wild` and `+4`) are color-agnostic by the rules of the game, so when the CNN predicts one of those we skip the color step entirely and emit the symbol on its own.

## Symbol Recognition with a CNN

Recognising the digits and action symbols (`0`–`9`, `+2`, `+4`, `wild`, `skip`, `reverse` — 15 classes total) is the part of the pipeline where a learned model clearly outperforms a classical approach, because of the visual variety of the symbols and of the printing/lighting conditions.

We trained a small VGG-style CNN (`hybric_detection_utils/train_uno.py`, class `UnoSymbolCNN`): four convolutional blocks with batch normalisation, global average pooling and a linear head, for about 4.8M parameters — well under the 12M cap of the competition. Inputs are 80x80 RGB crops normalised with the standard ImageNet mean/std. The training crops are generated by the same corner-extraction routine used at inference (`extract_corner_crops`), which guarantees that train and test distributions match.

At inference time (`src/card_labeler.predict_card`) we run the CNN on the two corner crops of each card (top-left and bottom-right corner indices). UNO cards repeat the same symbol in both corners, so we keep the prediction with the higher softmax confidence. The resulting symbol is then concatenated with the color label from the HSV pipeline (`submission_label` in `src/class_maps.py`) to form the final card string.

## Player Zone Assignment

Once all cards are detected, we still need to decide which card belongs to which player and which one is the center card. This is handled by `src/player_zones.py`.

The four player zones are not hard-coded in pixels but loaded from `data/jetons_et_backgrounds/backgrounds/bbox_player_position.csv`, which stores `(posx, posy, width, height)` for each player on the reference background. We convert those rectangles into **fractional coordinates** (relative to the reference image width and height) so the same zones can be reused on any image size — assignment then only depends on a card's centroid expressed as a fraction of the image.

The **center ROI** is derived geometrically from the four player zones: it spans horizontally between player 4's right edge and player 2's left edge, and vertically between player 3's bottom edge and player 1's top edge — i.e. the empty region in the middle of the table. If that computation ever degenerates (e.g. inconsistent CSV), we fall back to a centered 40%-60% rectangle.

The full assignment in `process_image_with_detections` works as follows:
1. `detect_center` keeps the unique detection whose centroid is inside the center ROI — if zero or more than one card falls there, we declare no center card (empty `center_card`).
2. All remaining detections are passed to `assign_hands`, which tests each centroid against the four player rectangles and groups detections by player.
3. Each per-player list is labelled by the CNN+color pipeline and joined with `;` into the final `player_X_cards` field; if a player has no card we emit `"EMPTY"`, matching the submission format.

## Active Player Detection

In order to detect the active player from an image, we decided that a classical approach would be the best. This is because the two tokens have a very distinct shape and color from anything else in the image, making them more straightforward to find compared to the cards.

The approach we decided to go for was to first threshold based on HSV values of the image to generate a mask of pixels that have a similar HSV values to the token, followed by some simple morphological operators, where we first open and then close. This ensures that we get nice smooth mask blobs and that small specs do not affect it meaningfully. Once that is done, we find the contour of all remaining blobs in the mask, and compute how close they are to the token we are looking for through a series of filters. The contour that is the closest to what we expect from the token is decided to be the active player token. We find the center point of the contour and map it to the closest player to determine the active player.

An important thing to note is that there are two different active player tokens, one that is rectangular and black and one that is circular and yellow. The black token is found on images with a white background and the yellow token on images with the colorful leafy background. Each token has its own method of computation because of their different shape and color, so we need to determine which background we have in our image. We found that a clear indicator of which background is in our image is to look at mean saturation over the entire image, where there is a noticeable difference. We use this to determine which token we should look for and then use the appropriate method to find it.